In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# EDA

# DEPENDENCIES

In [ ]:
!pip install -q -r /kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/requirements.txt

# INDEXING

## Pre-loader

In [ ]:
import json
from pathlib import Path

In [ ]:
folder = Path('/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/LegalQA - Public Test-20260827T072335Z-1-001/LegalQA - Public Test/selected-contexts')
docs = {}

for file in folder.rglob('*.json'):
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    docs[data['id']] = {
        'link': data.get('link'),
        'name': data.get('name'),
        'passage': data.get('passage'),
        'id': data['id'],
    }
    
with open('/kaggle/working/selected-contexts.json', 'w', encoding='utf-8') as f:
    json.dump(docs, f, ensure_ascii=False)

## Loader

In [ ]:
import json
from langchain_core.documents import Document

In [ ]:
def load_questions():
    with open('/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/LegalQA - Public Test-20260827T072335Z-1-001/LegalQA - Public Test/public-official.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    questions = []
    
    for qid, item in data.items():
        questions.append(
            Document(
                page_content=item['question'],
                metadata={
                    'id': qid,
                },
                id=qid
            )
        )
        
    return questions

In [ ]:
def load_answers():
    with open('/kaggle/input/datasets/vuwspace/legalqa-tr-li-cu-hi-php-lut-ting-vit/selected-contexts.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    answers = []
    
    for pid, item in data.items():
        answers.append(
            Document(
                page_content=item['passage'],
                metadata={
                    'link': f"{item['link']}",
                    'name': f"{item['name']}",
                    'id': pid,
                },
                id=pid
            )
        )
        
    return answers

## Chunkings

In [ ]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_core.embeddings import Embeddings
# from langchain_experimental.text_splitter import SemanticChunker
# from FlagEmbedding import BGEM3FlagModel

In [ ]:
# class bge_adapter(Embeddings):
#     def __init__(self, model=None):
#         self.model = model or BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
        
#     def embed_documents(self, texts):
#         return self.model.encode(texts, max_length=512)['dense_vecs'].tolist()
        
#     def embed_query(self, text):
#         return self.model.encode([text], max_length=512)['dense_vecs'][0].tolist()

In [ ]:
def split_doc(doc):
    text = doc.page_content
    parts = re.split(r'(?=(?:^|\n)\s*Điều\s+\d+[a-zđ]?\s*(?:[.:]|\n))', text)
    docs = []
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
            
        match = re.match(r'Điều\s+(\d+)', part)
        article = match.group(1) if match else None
        
        docs.append(
            Document(
                page_content=part,
                metadata={
                    **doc.metadata,
                    'article': article,
                }
            )
        )
        
    return docs if docs else [doc]

In [ ]:
def chunking():
    docs = load_answers()
    articles = []
    
    for d in docs:
        articles.extend(split_doc(d))
        
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=200,
        strip_whitespace=True,
        length_function=len,
        separators=['\n\nKhoản ', '\n\n', '\n', '. ', ' ', '',],
    )
    
    chunks = text_splitter.split_documents(articles)
    return chunks

In [ ]:
# def chunking():
#     docs = load_answers()
#     articles = []

#     for d in docs:
#         articles.extend(split_doc(d))

#     text_splitter = RecursiveCharacterTextSplitter(
#         chunk_size=1200,
#         chunk_overlap=200,
#         strip_whitespace=True,
#         length_function=len,
#         separators=['\n\nKhoản ', '\n\n', '\n', '. ', ' ', '',],
#     )

#     CHUNK_SIZE_LIMIT = 1200
#     # SEMANTIC_THRESHOLD = 3000
#     # semantic_splitter = None
#     chunks = []

#     for a in articles:
#         if len(a.page_content) <= CHUNK_SIZE_LIMIT:
#             chunks.append(a)
#         # elif len(a.page_content) <= SEMANTIC_THRESHOLD:
#         #     chunks.extend(text_splitter.split_documents([a]))
#         else:
#             chunks.extend(text_splitter.split_documents([a]))
#             # if semantic_splitter is None:
#             #     embeddings = bge_adapter()
#             #     semantic_splitter = SemanticChunker(
#             #         embeddings,
#             #         breakpoint_threshold_type='percentile',
#             #         breakpoint_threshold_amount=90,
#             #     )

#             # try:
#                 # sub_docs = semantic_splitter.create_documents(
#                 #     [a.page_content],
#                 #     metadatas=[a.metadata],
#                 # )
#                 # sub_docs = text_splitter.split_documents([a])
#             # except Exception:
#             #     sub_docs = [a]

#             # for sd in sub_docs:
#             #     if len(sd.page_content) <= CHUNK_SIZE_LIMIT:
#             #         chunks.append(sd)
#             #     else:
#             #         chunks.extend(text_splitter.split_documents([sd]))

#     chunks = [c for c in chunks if len(c.page_content.strip()) >= 30]
#     return chunks

In [ ]:
!mkdir -p /kaggle/working/faiss_index

In [ ]:
import os
import pickle

In [ ]:
INDEX_DIR = '/kaggle/working/faiss_index'
CHUNKS_PATH = os.path.join(INDEX_DIR, 'chunks.pkl')

In [ ]:
chunks = chunking()
os.makedirs(INDEX_DIR, exist_ok=True)

with open(CHUNKS_PATH, 'wb') as f:
    pickle.dump(chunks, f)

## Embeddings

In [ ]:
from FlagEmbedding import BGEM3FlagModel

In [ ]:
def bge_m3():
    return BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

In [ ]:
def embeddings():
    chunks = chunking()
    model = bge_m3()
    answer_sentences = [chunk.page_content for chunk in chunks]
    
    vectors = model.encode(
        answer_sentences,
        batch_size=32,
        max_length=8192,
        )['dense_vecs']
    
    return vectors

In [ ]:
import os
import numpy as np

In [ ]:
INDEX_DIR = '/kaggle/working/faiss_index'
EMBEDDS_PATH = os.path.join(INDEX_DIR, 'embedds.npy')

In [ ]:
vectors = embeddings()
os.makedirs(INDEX_DIR, exist_ok=True)

np.save(EMBEDDS_PATH, vectors)

## Vector Store

In [ ]:
import os
import pickle
import numpy as np
import faiss

In [ ]:
INDEX_DIR = '/kaggle/working/faiss_index'
EMBEDDS_PATH = os.path.join(INDEX_DIR, 'embedds.npy')
INDEX_PATH = os.path.join(INDEX_DIR, 'index.faiss')

In [ ]:
def vectorstore(vectors):
    idx = faiss.IndexFlatIP(1024)
    os.makedirs(INDEX_DIR, exist_ok=True)
    
    vectors = vectors.astype('float32')
    faiss.normalize_L2(vectors)
    
    idx.add(vectors)
    faiss.write_index(idx, INDEX_PATH)
    
    return idx

In [ ]:
vectors = np.load(EMBEDDS_PATH)
idx = vectorstore(vectors)

# QUERY

## Retriever

In [ ]:
import faiss
import numpy as np

In [ ]:
INDEX_DIR = '/kaggle/working/faiss_index'
CHUNKS_PATH = os.path.join(INDEX_DIR, 'chunks.pkl')

In [ ]:
idx = faiss.read_index(INDEX_PATH)

In [ ]:
def get_chunks():
    with open(CHUNKS_PATH, 'rb') as f:
        return pickle.load(f)

In [ ]:
chunks = get_chunks()
model = bge_m3()

def retriever(idx, chunks, model):
    def retrieve(question, k=20, top_n=5, use_rerank=True):
        q_vector = model.encode(
            [question],
            max_length=8192
            )['dense_vecs']
        q_vector = np.array(q_vector).astype('float32')
        faiss.normalize_L2(q_vector)
        
        scores, indices = idx.search(q_vector, k)
        candidates = [chunks[i] for i in indices[0] if i != -1]
        
        if not use_rerank:
            return candidates[:top_n]
        return rerank(question, candidates, top_n=top_n)
        
    def retrieve_batch(questions, k=20, top_n=5, batch_size=32, use_rerank=True):
        q_vectors = model.encode(
            questions,
            batch_size=batch_size,
            max_length=8192
            )['dense_vecs']
        q_vectors = np.array(q_vectors).astype('float32')
        faiss.normalize_L2(q_vectors)
        
        scores, indices = idx.search(q_vectors, k)
        all_candidates = []
        
        for i in range(len(questions)):
            candidates = [chunks[j] for j in indices[i] if j != -1]
            all_candidates.append(candidates)
            
        if not use_rerank:
            return [c[:top_n] for c in all_candidates]
            
        return rerank_batch(questions, all_candidates, top_n=top_n)
        
    return retrieve, retrieve_batch

## Re-ranker

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
_reranker = None

def get_reranker():
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')
    return _reranker
    
def rerank(question, candidates, top_n=5):
    if not candidates:
        return []
        
    reranker = get_reranker()
    pairs = [[question, c.page_content] for c in candidates]
    scores = reranker.predict(pairs)
    scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    
    return [doc for doc, score in scored[:top_n]]

def rerank_batch(questions, candidates_list, top_n=5):
    reranker = get_reranker()
    results = []
    
    for question, candidates in zip(questions, candidates_list):
        if not candidates:
            results.append([])
            continue
            
        pairs = [[question, c.page_content] for c in candidates]
        scores = reranker.predict(pairs)
        scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        results.append([doc for doc, score in scored[:top_n]])
        
    return results

## Contexts

In [ ]:
import os
import json

In [ ]:
CONTEXTS_PATH = '/kaggle/working/contexts.json'

def contexts(output_path=CONTEXTS_PATH, k=20, top_n=5, use_rerank=True, batch_size=32):
    questions = load_questions()
    existing = {}
    
    if os.path.exists(output_path):
        with open(output_path, 'r', encoding='utf-8') as f:
            existing = json.load(f)
            
    pending = [q for q in questions if str(q.metadata.get('id')) not in existing]
    
    if not pending:
        print('Đã retrieve đủ context cho toàn bộ câu hỏi.')
        return
        
    print(f'Cần retrieve context cho {len(pending)}/{len(questions)} câu hỏi.')

    p_ids = [str(p.metadata.get('id')) for p in pending]
    p_questions = [p.page_content.strip() for p in pending]
    p_questions_search = [q.lower() for q in p_questions]

    chunks = get_chunks()
    model = bge_m3()
    
    r, r_batch = retriever(idx, chunks, model)
    
    for i in range(0, len(p_questions), batch_size):
        b_ids = p_ids[i: i + batch_size]
        b_questions = p_questions[i: i + batch_size]
        b_questions_search = p_questions_search[i: i + batch_size]
        b_docs = r_batch(b_questions_search, k=k, top_n=top_n, use_rerank=use_rerank)

        for qid, question, docs in zip(b_ids, b_questions, b_docs):
            context = '\n\n'.join(d.page_content for d in docs)
            existing[qid] = {
                'question': question,
                'context': context,
                }

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(existing, f, ensure_ascii=False)

        print(f'Đã retrieve {min(i + batch_size, len(p_questions))}/{len(p_questions)}')
    print(f'\nHoàn tất retrieve. Đã lưu context cho {len(existing)}/{len(questions)} câu vào {output_path}.')

if __name__=='__main__':
    contexts()

## LLM

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [2]:
SYSTEM_PROMPT = (
    'Bạn là một trợ lý nghiêm ngặt, tập trung vào việc trích dẫn, dành cho một cơ sở kiến thức riêng tư.\n'
    'QUY TẮC:\n'
    '1. CHỈ sử dụng ngữ cảnh được cung cấp để trả lời.\n'
    '2. Nếu câu trả lời không được nêu rõ ràng trong ngữ cảnh, hãy trả lời CHÍNH XÁC câu này và không thêm bất kỳ nội dung nào khác: "Tôi không biết dựa trên các tài liệu được cung cấp."\n'
    '3. KHÔNG sử dụng kiến thức bên ngoài, suy đoán hoặc thông tin từ web.\n'
    '4. CHỈ trả lời bằng tiếng Việt. Không được bao gồm bất kỳ nội dung tiếng Anh nào, phần mở đầu bằng tiếng Anh hoặc câu dẫn bằng tiếng Anh (ví dụ: KHÔNG viết "The answer is", "Based on the context", "The provided text states" hoặc các câu tương tự). Không được lặp lại hoặc diễn đạt lại câu hỏi.\n'
    '5. CHỈ xuất trực tiếp phần câu trả lời cuối cùng bằng tiếng Việt — không có lời giới thiệu, không có bình luận mang tính meta, không giải thích về những gì bạn đang làm.\n'
    '6. Khi ngữ cảnh có chứa số điều/khoản của văn bản pháp luật (ví dụ: "Điều 5", "Khoản 2"), hãy trích dẫn chúng một cách tự nhiên trong câu trả lời, ví dụ: "Theo Điều 5 Khoản 2..." — phù hợp với văn phong pháp lý được sử dụng trong ngữ cảnh, không diễn đạt lại theo cách nói thông thường.\n'
    '7. Cung cấp câu trả lời đầy đủ, bao gồm căn cứ pháp lý có liên quan từ ngữ cảnh, không chỉ đưa ra một thông tin đơn lẻ — trừ khi câu hỏi yêu cầu rõ ràng một giá trị ngắn duy nhất.\n\n'
)

In [3]:
def load_model(device_map=None):
    model_name = "Qwen/Qwen2.5-3B-Instruct"
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map=device_map or 'auto',
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
    model.eval()
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer
    
def generate(model, tokenizer, question, context):
    answers = generate_batch(model, tokenizer, [question], [context])
    return answers[0]

@torch.no_grad()
def generate_batch(model, tokenizer, questions, contexts):
    messages_list = [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {q}"},
        ]
        for q, ctx in zip(questions, contexts)
    ]
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_list
    ]
    
    tokenizer.padding_side = "left"
    model_inputs = tokenizer(
        texts, return_tensors="pt", padding=True,
        truncation=True, max_length=3072
    )
    model_inputs = {k: v.to(next(model.parameters()).device) for k, v in model_inputs.items()}
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    results = []
    
    for input_ids, output_ids in zip(model_inputs['input_ids'], generated_ids):
        new_tokens = output_ids[len(input_ids):]
        results.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
        
    return results

## Submission

### Test 1 câu

In [ ]:
# def rag_pipeline():
#     print('Đang load retriever và model...\n')
#     idx, chunks, model = vectorstore()
#     r, _ = retriever(idx, chunks, model)
#     model, tokenizer = load_model()

#     print('\nSẵn sàng, gõ "exit" để thoát chương trình.\n')

#     while True:
#         question = input('Question: ').strip().lower()
#         if question == 'exit':
#             break
#         print()

#         docs = r(question, k=20, top_n=5, use_rerank=True)
#         context = '\n\n'.join(d.page_content for d in docs)
#         answer = generate(model, tokenizer, question, context)
#         print(f'\nAnswer: {answer}\n')

# if __name__ == "__main__":
#     rag_pipeline()

In [4]:
import os
import json
import time
import gc
import torch

In [5]:
CONTEXTS_PATH = '/kaggle/working/contexts.json'
BATCH_SIZE = 8

def load_existing_submission(path='/kaggle/working/submission.json'):
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def make_submission(contexts_path=CONTEXTS_PATH, output_path='/kaggle/working/submission.json', batch_size=BATCH_SIZE, device_map=None):
    with open(contexts_path, 'r', encoding='utf-8') as f:
        contexts = json.load(f)
        
    submission = load_existing_submission(output_path)
    p_ids = [qid for qid in contexts if qid not in submission]
    
    if not p_ids:
        print('Không còn câu nào cần xử lý.')
        return

    print(f'Cần xử lý {len(p_ids)}/{len(contexts)} câu trả lời.')
    print(f'Batch size: {batch_size}')
    print()
    
    for var_name in ['model', '_reranker']:
        if var_name in globals() and globals()[var_name] is not None:
            free_gpu(globals()[var_name])
            globals()[var_name] = None
            
    gc.collect()
    torch.cuda.empty_cache()
    
    model, tokenizer = load_model(device_map=device_map)
    start = time.time()
    total_done = 0

    for i in range(0, len(p_ids), batch_size):
        b_ids = p_ids[i: i + batch_size]
        b_questions = [contexts[qid]['question'] for qid in b_ids]
        b_contexts = [contexts[qid]['context'] for qid in b_ids]

        try:
            b_answers = generate_batch(model, tokenizer, b_questions, b_contexts)
        except torch.cuda.OutOfMemoryError as e:
            print(f'OOM batch {b_ids[0]}: {e}')
            
            torch.cuda.empty_cache()
            b_answers = []
            
            for q, c in zip(b_questions, b_contexts):
                try:
                    a = generate_batch(model, tokenizer, [q], [c])[0]
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    a = 'Tôi không biết dựa trên các tài liệu được cung cấp.'
                
                b_answers.append(a)

        for qid, answer in zip(b_ids, b_answers):
            submission[qid] = {'answer': answer}

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(submission, f, ensure_ascii=False, indent=4)

        total_done += len(b_ids)
        elapsed = time.time() - start
        avg = elapsed / total_done
        remaining = avg * (len(p_ids) - total_done)
        print(
            f'Đã xử lý {total_done}/{len(p_ids)} '
            f'({b_ids[0]}...{b_ids[-1]}) — còn ~{remaining / 60:.1f} phút')
    print(f'\nHoàn tất. Tổng cộng {len(submission)}/{len(contexts)} câu đã xử lý.')

if __name__ == "__main__":
    make_submission()

Cần xử lý 1000/1000 câu trả lời.
Batch size: 8



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Đã xử lý 8/1000 (80189...61335) — còn ~528.0 phút
Đã xử lý 16/1000 (104595...13409) — còn ~382.8 phút
Đã xử lý 24/1000 (134019...99819) — còn ~403.5 phút
Đã xử lý 32/1000 (140571...21043) — còn ~425.9 phút
Đã xử lý 40/1000 (97355...162669) — còn ~431.0 phút
Đã xử lý 48/1000 (133691...113581) — còn ~429.3 phút
Đã xử lý 56/1000 (102339...151429) — còn ~424.1 phút
Đã xử lý 64/1000 (19109...39407) — còn ~406.7 phút
Đã xử lý 72/1000 (80125...61955) — còn ~402.3 phút
Đã xử lý 80/1000 (107143...8173) — còn ~403.9 phút
Đã xử lý 88/1000 (49975...113931) — còn ~396.4 phút
Đã xử lý 96/1000 (64783...124851) — còn ~388.2 phút
Đã xử lý 104/1000 (58173...73195) — còn ~399.8 phút
Đã xử lý 112/1000 (17727...72801) — còn ~403.6 phút
Đã xử lý 120/1000 (15029...123443) — còn ~408.4 phút
Đã xử lý 128/1000 (75131...65771) — còn ~402.5 phút
Đã xử lý 136/1000 (74191...72539) — còn ~398.0 phút
Đã xử lý 144/1000 (10013...38583) — còn ~383.7 phút
Đã xử lý 152/1000 (156629...11675) — còn ~383.8 phút
Đã xử lý 160/